# 3D benchmark from Schäfer+Turek

https://wwwold.mathematik.tu-dortmund.de/lsiii/cms/papers/SchaeferTurek1996.pdf

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import ipywidgets as widgets

In [ ]:
box = Box((0,0,0), (2.5,0.41,0.41))
box.faces.name="wall"
box.faces.Min(X).name="inlet"
box.faces.Max(X).name="outlet"
cyl = Cylinder((0.5,0.2,0), Z, h=0.41,r=0.05)
cyl.faces.name="cyl"
shape = box-cyl

# maxh = 0.1
# order = 2

maxh = 0.1
order = 2


mesh = Mesh(OCCGeometry(shape).GenerateMesh(maxh=maxh)).Curve(order)
print (mesh.GetNE(VOL))

In [ ]:
Draw (mesh);

Navier-Stokes solver using Incremental Pressure Correction Scheme (IPCS)

In [ ]:
from NavierStokesIPCS_MCS import NavierStokes

In [ ]:
# original benchmark:
# um=2.25
# Reynolds 1000
um = 22.5  
timestep = 1e-2/um

uin = (um*16*y*(0.41-y)*z*(0.41-z)/0.41**4,0, 0)

with TaskManager(pajetrace=10**9):
    solver = NavierStokes(mesh, nu=1e-3, inflow="inlet", outflow="outlet", wall="wall|cyl",
                          uin=uin, timestep=timestep, substeps=10, order=order, verbose=1)                    

In [ ]:
with TaskManager(pajetrace=10**8):
    solver.SolveInitial()

In [ ]:
clipping = { "function" : True,  "pnt" : (2.5,0.2,0.2), "vec" : (0,0,-1) }

scene = Draw (solver.velocity, clipping=clipping, order=order, min=0, max=um);
scenep = Draw (solver.pressure, mesh, clipping=clipping, order=order, draw_surf=False, min=-100, max=100);

tw = widgets.Text(value='t = 0')
display(tw)

In [ ]:
# tend = 10
tend = 100*timestep
t = 0
with TaskManager(): # pajetrace=10**9):
    while t < tend:
        t += timestep
        solver.DoTimeStep()
        # print (step*timestep)
        tw.value = "t = " + str(t)
        scene.Redraw()
        scenep.Redraw()

The solver: [NavierStokesIPCS_MCS.py](NavierStokesIPCS_MCS.py)